In [2]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.0/128.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.2/626.2 kB 37.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import ccxt
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import time
import gspread
from google.oauth2.service_account import Credentials
import warnings
import requests
import io
from matplotlib import gridspec

# Suppress specific future warnings from pandas to keep console clean
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- Telegram Configuration ---
TELEGRAM_TOKEN = '7806611781:AAEAvJZiH296k_4lyfCBuFYwIxMBCJJjR6w'
TELEGRAM_CHAT_ID = '790771042'

# --- Konfigurasi Awal Bot ---
PAIRS = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'XRP/USDT', 'SUI/USDT',
         'HAEDAL/USDT', 'ADA/USDT', 'DOGE/USDT', 'PEPE/USDT', 'TRUMP/USDT',
         'MOODENG/USDT', 'FARTCOIN/USDT','HYPE/USDT','HYPER/USDT','BNB/USDT',
         'LTC/USDT', 'AVAX/USDT', 'NEAR/USDT','TRX/USDT','ONDO/USDT']
TIMEFRAME = '15m' # Timeframe untuk data candlestick (e.g., '15m', '1h', '4h', '1d')
RSI_PERIOD = 14
OVERBOUGHT_RSI = 65
OVERSOLD_RSI = 35
EMA_FAST = 9
EMA_SLOW = 21
ATR_PERIOD = 10
ATR_MULTIPLIER_SL = 1.5 # Multiplier untuk Stop Loss berdasarkan ATR
ATR_MULTIPLIER_TP = 3.0 # Multiplier untuk Take Profit berdasarkan ATR
COMMISSION_PER_TRADE_PERCENT = 0.00075 # Contoh: KuCoin Spot Trading Fee (Maker/Taker) 0.075%

# Fibonacci levels for partial take profits
FIB_LEVELS = [0.236, 0.618, 0.786]

# Dummy Wallet untuk Backtesting
INITIAL_USDT_BALANCE = 100 # Total modal awal untuk semua pair dalam backtest

# --- Konfigurasi Google Sheets ---
GOOGLE_SHEETS_KEY_PATH = 'google_sheets_key.json'
GOOGLE_SHEET_NAME = 'Trading Bot Logs' # Nama Google Sheet Anda
BACKTEST_WORKSHEET_NAME = 'Backtest Transactions' # Nama worksheet untuk histori backtest
LIVE_SIGNALS_WORKSHEET_NAME = 'Live Signals' # Nama worksheet untuk sinyal live

# Setup objek exchange KuCoin
exchange = ccxt.kucoin({
    'enableRateLimit': True,
    # 'apiKey': 'YOUR_API_KEY',
    # 'secret': 'YOUR_SECRET_KEY',
    # 'password': 'YOUR_PASSWORD',
})

# --- Telegram Functions ---
def escape_markdown(text):
    """Escape all MarkdownV2 reserved characters."""
    reserved_chars = ['_', '*', '[', ']', '(', ')', '~', '`', '>', '#', '+', '-', '=', '|', '{', '}', '.', '!']
    for char in reserved_chars:
        text = text.replace(char, f'\\{char}')
    return text

def send_telegram_message(text):
    """Send text message to Telegram with proper MarkdownV2 escaping."""
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"

    # Escape ALL reserved characters
    text = escape_markdown(text)

    params = {
        'chat_id': TELEGRAM_CHAT_ID,
        'text': text,
        'parse_mode': 'MarkdownV2'
    }
    try:
        response = requests.post(url, params=params, timeout=10)
        response.raise_for_status()
        return True
    except Exception as e:
        print(f"Telegram API Error: {str(e)}")
        print(f"Response: {response.text if 'response' in locals() else 'None'}")
        return False

def send_telegram_image(caption, image_buffer):
    """Send image to Telegram with escaped caption."""
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendPhoto"

    try:
        image_buffer.seek(0)
        files = {'photo': ('chart.png', image_buffer, 'image/png')}
        data = {
            'chat_id': TELEGRAM_CHAT_ID,
            'caption': escape_markdown(caption[:1024]),  # Escape caption text
        }
        response = requests.post(url, files=files, data=data, timeout=15)
        response.raise_for_status()
        return True
    except Exception as e:
        print(f"Telegram Image Error: {str(e)}")
        return False

# --- Chart Visualization Functions ---
def visualize_trade_setup(df, symbol, direction, entry_price, stop_loss, take_profit, atr_value):
    """Create a visualization of the trade setup with Fibonacci TP levels."""
    # Create figure with custom layout
    plt.figure(figsize=(12, 10))
    gs = gridspec.GridSpec(3, 1, height_ratios=[3, 1, 1])

    # Price chart
    ax1 = plt.subplot(gs[0])
    ax1.set_title(f'{symbol} - {direction.upper()} Setup')

    # Plot candles
    last_50 = df.iloc[-50:]
    for idx, row in last_50.iterrows():
        color = 'green' if row['close'] >= row['open'] else 'red'
        ax1.plot([idx, idx], [row['low'], row['high']], color=color, linewidth=1)
        ax1.plot([idx, idx], [row['open'], row['close']], color=color, linewidth=3)

    # Calculate Fibonacci TP levels
    if direction == 'buy':
        fib_levels = [entry_price + (take_profit - entry_price) * level for level in FIB_LEVELS]
    else:
        fib_levels = [entry_price - (entry_price - take_profit) * level for level in FIB_LEVELS]

    # Plot entry, SL, TP and Fib levels
    ax1.axhline(entry_price, color='blue', linestyle='--', label=f'Entry: {entry_price:.4f}')
    ax1.axhline(stop_loss, color='red', linestyle='--', label=f'SL: {stop_loss:.4f}')
    ax1.axhline(take_profit, color='green', linestyle='--', label=f'Final TP: {take_profit:.4f}')

    # Plot Fibonacci TP levels
    colors = ['orange', 'purple', 'cyan']
    for i, level in enumerate(fib_levels):
        ax1.axhline(level, color=colors[i], linestyle=':',
                   label=f'TP {FIB_LEVELS[i]*100:.1f}%: {level:.4f}')

    # Plot EMAs
    ax1.plot(last_50.index, last_50['ema_fast'], label=f'EMA{EMA_FAST}', color='lime', alpha=0.7)
    ax1.plot(last_50.index, last_50['ema_slow'], label=f'EMA{EMA_SLOW}', color='magenta', alpha=0.7)

    ax1.legend()
    ax1.grid(True)

    # RSI chart
    ax2 = plt.subplot(gs[1], sharex=ax1)
    ax2.plot(last_50.index, last_50['rsi'], label='RSI', color='purple')
    ax2.axhline(OVERBOUGHT_RSI, color='red', linestyle='--')
    ax2.axhline(OVERSOLD_RSI, color='green', linestyle='--')
    ax2.set_ylabel('RSI')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()

    # Save to buffer
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100)
    buf.seek(0)
    plt.close()

    return buf

# --- Modified Backtest Engine for Fibonacci TP ---
def backtest(df_combined, initial_balance=INITIAL_USDT_BALANCE, risk_per_trade_percent=0.01,
             atr_multiplier_sl=ATR_MULTIPLIER_SL, atr_multiplier_tp=ATR_MULTIPLIER_TP,
             commission_per_trade_percent=COMMISSION_PER_TRADE_PERCENT):
    """Menjalankan simulasi backtest pada data historis dengan Fibonacci TP."""
    balance = initial_balance
    entries = []
    trade_count = 0
    open_positions = {}

    required_cols = ['close', 'high', 'low', 'atr', 'rsi', 'ema_fast', 'ema_slow',
                    'long_signal', 'short_signal', 'symbol', 'good_time']

    df_cleaned = df_combined.dropna(subset=required_cols).copy()

    if df_cleaned.empty:
        print("Tidak cukup data valid setelah perhitungan indikator untuk backtesting.")
        return pd.DataFrame(), initial_balance, pd.DataFrame()

    grouped_by_time = df_cleaned.groupby(df_cleaned.index)
    balance_history = []

    for timestamp, current_bar_data_for_all_symbols in grouped_by_time:
        # Close positions first
        symbols_to_close = []
        for symbol_in_pos, position in list(open_positions.items()):
            current_symbol_bar = current_bar_data_for_all_symbols[current_bar_data_for_all_symbols['symbol'] == symbol_in_pos]

            if current_symbol_bar.empty:
                continue

            current_high = current_symbol_bar['high'].iloc[0]
            current_low = current_symbol_bar['low'].iloc[0]
            current_close = current_symbol_bar['close'].iloc[0]

            exit_price = None
            exit_type = None
            partial_tp = False

            if position['direction'] == 'buy':
                # Check for partial TPs first
                for i, tp_level in enumerate(position['fib_tp_levels']):
                    if current_high >= tp_level and not position['tp_hit'][i]:
                        exit_price = tp_level
                        exit_type = f'TP {FIB_LEVELS[i]*100:.1f}%'
                        position['tp_hit'][i] = True
                        partial_tp = True
                        break

                # Then check for SL or final TP
                if not partial_tp:
                    if current_low <= position['stop_loss']:
                        exit_price = position['stop_loss']
                        exit_type = 'SL'
                    elif current_high >= position['take_profit']:
                        exit_price = position['take_profit']
                        exit_type = 'Final TP'
                    elif current_symbol_bar['short_signal'].iloc[0] == 1 and current_symbol_bar['long_signal'].iloc[0] == 0:
                        exit_price = current_close
                        exit_type = 'Counter-Signal'
            else:  # sell position
                # Check for partial TPs first
                for i, tp_level in enumerate(position['fib_tp_levels']):
                    if current_low <= tp_level and not position['tp_hit'][i]:
                        exit_price = tp_level
                        exit_type = f'TP {FIB_LEVELS[i]*100:.1f}%'
                        position['tp_hit'][i] = True
                        partial_tp = True
                        break

                # Then check for SL or final TP
                if not partial_tp:
                    if current_high >= position['stop_loss']:
                        exit_price = position['stop_loss']
                        exit_type = 'SL'
                    elif current_low <= position['take_profit']:
                        exit_price = position['take_profit']
                        exit_type = 'Final TP'
                    elif current_symbol_bar['long_signal'].iloc[0] == 1 and current_symbol_bar['short_signal'].iloc[0] == 0:
                        exit_price = current_close
                        exit_type = 'Counter-Signal'

            if exit_price is not None:
                if position['direction'] == 'buy':
                    pnl = (exit_price - position['entry_price']) * position['lot_size_amount']
                else:
                    pnl = (position['entry_price'] - exit_price) * position['lot_size_amount']

                commission_amount = (position['entry_price'] * position['lot_size_amount'] * commission_per_trade_percent) + \
                                    (exit_price * position['lot_size_amount'] * commission_per_trade_percent)
                pnl -= commission_amount

                balance += pnl
                trade_count += 1

                sl_distance_actual = abs(position['entry_price'] - position['stop_loss'])
                tp_distance_actual = abs(position['take_profit'] - position['entry_price'])
                risk_reward_ratio = tp_distance_actual / sl_distance_actual if sl_distance_actual != 0 else np.nan

                entries.append({
                    'entry_date': position['entry_date'],
                    'entry_price': position['entry_price'],
                    'exit_date': timestamp,
                    'exit_price': exit_price,
                    'direction': position['direction'],
                    'pnl': pnl,
                    'balance': balance,
                    'exit_type': exit_type,
                    'sl_distance': sl_distance_actual,
                    'tp_distance': tp_distance_actual,
                    'risk_reward': risk_reward_ratio,
                    'commission': commission_amount,
                    'lot_size_amount': position['lot_size_amount'],
                    'symbol': symbol_in_pos
                })

                if partial_tp:
                    # Update position size for remaining TPs
                    position['lot_size_amount'] *= 0.5  # Example: take 50% profit at each TP level
                else:
                    symbols_to_close.append(symbol_in_pos)

        for symbol_to_remove in symbols_to_close:
            del open_positions[symbol_to_remove]

        # Open new positions
        for _, row_for_symbol in current_bar_data_for_all_symbols.iterrows():
            current_symbol = row_for_symbol['symbol']

            if current_symbol in open_positions or row_for_symbol['good_time'] == 0:
                continue

            next_bar_candidates = df_cleaned[
                (df_cleaned.index > timestamp) &
                (df_cleaned['symbol'] == current_symbol)
            ].sort_index()

            if next_bar_candidates.empty:
                continue

            next_bar_for_symbol = next_bar_candidates.iloc[0]
            entry_price = next_bar_for_symbol['open']
            current_atr = row_for_symbol['atr']

            if pd.isna(current_atr) or current_atr <= 0.0001:
                continue

            sl_distance = current_atr * atr_multiplier_sl
            tp_distance = current_atr * atr_multiplier_tp
            risk_dollars_per_trade = balance * risk_per_trade_percent

            if sl_distance == 0:
                continue

            lot_size_amount = risk_dollars_per_trade / sl_distance
            lot_size_amount = round(lot_size_amount, 8)

            if lot_size_amount < 0.00000001:
                continue

            if balance < (entry_price * lot_size_amount * (1 + commission_per_trade_percent)):
                continue

            if row_for_symbol['long_signal'] == 1 and row_for_symbol['short_signal'] == 0:
                direction = "buy"
                sl = entry_price - sl_distance
                tp = entry_price + tp_distance

                if sl >= entry_price or tp <= entry_price:
                    continue

                # Calculate Fibonacci TP levels
                fib_tp_levels = [entry_price + (tp - entry_price) * level for level in FIB_LEVELS]

                open_positions[current_symbol] = {
                    'entry_date': timestamp,
                    'entry_price': entry_price,
                    'direction': direction,
                    'stop_loss': sl,
                    'take_profit': tp,
                    'fib_tp_levels': fib_tp_levels,
                    'tp_hit': [False] * len(FIB_LEVELS),
                    'lot_size_amount': lot_size_amount,
                }

            elif row_for_symbol['short_signal'] == 1 and row_for_symbol['long_signal'] == 0:
                direction = "sell"
                sl = entry_price + sl_distance
                tp = entry_price - tp_distance

                if sl <= entry_price or tp >= entry_price:
                    continue

                # Calculate Fibonacci TP levels
                fib_tp_levels = [entry_price - (entry_price - tp) * level for level in FIB_LEVELS]

                open_positions[current_symbol] = {
                    'entry_date': timestamp,
                    'entry_price': entry_price,
                    'direction': direction,
                    'stop_loss': sl,
                    'take_profit': tp,
                    'fib_tp_levels': fib_tp_levels,
                    'tp_hit': [False] * len(FIB_LEVELS),
                    'lot_size_amount': lot_size_amount,
                }

        balance_history.append({'date': timestamp, 'balance': balance})

    # Close remaining positions at the end
    for symbol, position in list(open_positions.items()):
        final_exit_price_bar = df_cleaned[df_cleaned['symbol'] == symbol].iloc[-1]
        final_exit_price = final_exit_price_bar['close']

        if position['direction'] == 'buy':
            pnl = (final_exit_price - position['entry_price']) * position['lot_size_amount']
        else:
            pnl = (position['entry_price'] - final_exit_price) * position['lot_size_amount']

        commission_amount = (position['entry_price'] * position['lot_size_amount'] * commission_per_trade_percent) + \
                            (final_exit_price * position['lot_size_amount'] * commission_per_trade_percent)
        pnl -= commission_amount
        balance += pnl
        trade_count += 1

        sl_distance_actual = abs(position['entry_price'] - position['stop_loss'])
        tp_distance_actual = abs(position['take_profit'] - position['entry_price'])
        risk_reward_ratio = tp_distance_actual / sl_distance_actual if sl_distance_actual != 0 else np.nan

        entries.append({
            'entry_date': position['entry_date'],
            'entry_price': position['entry_price'],
            'exit_date': final_exit_price_bar.name,
            'exit_price': final_exit_price,
            'direction': position['direction'],
            'pnl': pnl,
            'balance': balance,
            'exit_type': 'Closed at End',
            'sl_distance': sl_distance_actual,
            'tp_distance': tp_distance_actual,
            'risk_reward': risk_reward_ratio,
            'commission': commission_amount,
            'lot_size_amount': position['lot_size_amount'],
            'symbol': symbol
        })

    results_df = pd.DataFrame(entries).sort_values('entry_date')
    balance_history_df = pd.DataFrame(balance_history).sort_values('date')

    return results_df, balance, balance_history_df

# --- Modified Forward Test with Telegram Notifications ---
def forward_test():
    """Run forward test with Telegram notifications."""
    print("\n--- Memulai Mode Forward Test (Mengirim Sinyal Live ke Telegram) ---")
    print("Mode ini akan berjalan terus menerus. Tekan Ctrl+C untuk menghentikan.")

    sent_signals = set()

    interval_value = int(''.join(filter(str.isdigit, TIMEFRAME)))
    interval_unit = ''.join(filter(str.isalpha, TIMEFRAME)).lower()

    if interval_unit == 'm':
        sleep_duration = timedelta(minutes=interval_value)
    elif interval_unit == 'h':
        sleep_duration = timedelta(hours=interval_value)
    elif interval_unit == 'd':
        sleep_duration = timedelta(days=interval_value)
    else:
        print("Unit timeframe tidak didukung untuk durasi tidur. Default ke 4 jam.")
        sleep_duration = timedelta(hours=4)

    sleep_seconds = sleep_duration.total_seconds()

    while True:
        try:
            current_live_signals = []

            for pair in PAIRS:
                df_live = fetch_crypto_data_ccxt(pair, TIMEFRAME, limit=200)

                min_data_points = max(RSI_PERIOD, ATR_PERIOD) + 2
                if len(df_live) < min_data_points:
                    print(f"Tidak cukup data ({len(df_live)} bar) untuk {pair}. Lewati untuk saat ini.")
                    continue

                df_live = rsi_strategy(df_live)
                df_live = atr(df_live)
                df_live = calculate_ema(df_live, EMA_FAST, EMA_SLOW)
                df_live = candlestick_patterns(df_live)
                df_live = generate_signals(df_live)

                latest_bar = df_live.iloc[-1]
                latest_timestamp = latest_bar.name

                if latest_bar['long_signal'] == 1 and latest_bar['short_signal'] == 0:
                    signal_key = (latest_timestamp, pair, 'buy')
                    if signal_key not in sent_signals:
                        print(f"Sinyal BELI baru untuk {pair} pada {latest_timestamp}. Harga: {latest_bar['close']:.8f}")

                        # Calculate trade parameters
                        entry_price = latest_bar['close']
                        atr_value = latest_bar['atr']
                        stop_loss = entry_price - atr_value * ATR_MULTIPLIER_SL
                        take_profit = entry_price + atr_value * ATR_MULTIPLIER_TP

                        # Calculate Fibonacci TP levels
                        fib_tp_levels = [entry_price + (take_profit - entry_price) * level for level in FIB_LEVELS]

                        # Create visualization
                        chart_image = visualize_trade_setup(df_live, pair, 'buy', entry_price, stop_loss, take_profit, atr_value)

                        # Prepare Telegram message
                        message = (
                            f"🚀 *Sinyal BELI* untuk *{pair}*\n"
                            f"⏰ Waktu: {latest_timestamp.strftime('%Y-%m-%d %H:%M:%S')}\n"
                            f"💰 Harga: {entry_price:.8f}\n"
                            f"🛑 Stop Loss: {stop_loss:.8f}\n"
                            f"🎯 Take Profit: {take_profit:.8f}\n"
                            f"📊 ATR: {atr_value:.8f}\n\n"
                            f"📈 *Fibonacci TP Levels:*\n"
                            f"• TP1 (23.6%): {fib_tp_levels[0]:.8f}\n"
                            f"• TP2 (61.8%): {fib_tp_levels[1]:.8f}\n"
                            f"• TP3 (78.6%): {fib_tp_levels[2]:.8f}\n\n"
                            f"📊 *Indikator:*\n"
                            f"• RSI: {latest_bar['rsi']:.2f}\n"
                            f"• EMA{EMA_FAST}: {latest_bar['ema_fast']:.8f}\n"
                            f"• EMA{EMA_SLOW}: {latest_bar['ema_slow']:.8f}"
                        )

                        # Send to Telegram
                        send_telegram_message(message)
                        send_telegram_image("Trade Setup Visualization", chart_image)

                        # Add to Google Sheets (unchanged from original)
                        current_live_signals.append({
                            'timestamp': latest_timestamp.strftime('%Y-%m-%d %H:%M:%S'),
                            'symbol': pair,
                            'direction': 'buy',
                            'price': f"{latest_bar['close']:.8f}",
                            'signal_type': 'Long Signal',
                            'RSI': f"{latest_bar['rsi']:.2f}",
                            'EMA_Fast': f"{latest_bar['ema_fast']:.8f}",
                            'EMA_Slow': f"{latest_bar['ema_slow']:.8f}"
                        })
                        sent_signals.add(signal_key)

                if latest_bar['short_signal'] == 1 and latest_bar['long_signal'] == 0:
                    signal_key = (latest_timestamp, pair, 'sell')
                    if signal_key not in sent_signals:
                        print(f"Sinyal JUAL baru untuk {pair} pada {latest_timestamp}. Harga: {latest_bar['close']:.8f}")

                        # Calculate trade parameters
                        entry_price = latest_bar['close']
                        atr_value = latest_bar['atr']
                        stop_loss = entry_price + atr_value * ATR_MULTIPLIER_SL
                        take_profit = entry_price - atr_value * ATR_MULTIPLIER_TP

                        # Calculate Fibonacci TP levels
                        fib_tp_levels = [entry_price - (entry_price - take_profit) * level for level in FIB_LEVELS]

                        # Create visualization
                        chart_image = visualize_trade_setup(df_live, pair, 'sell', entry_price, stop_loss, take_profit, atr_value)

                        # Prepare Telegram message
                        message = (
                            f"🔻 *Sinyal JUAL* untuk *{pair}*\n"
                            f"⏰ Waktu: {latest_timestamp.strftime('%Y-%m-%d %H:%M:%S')}\n"
                            f"💰 Harga: {entry_price:.8f}\n"
                            f"🛑 Stop Loss: {stop_loss:.8f}\n"
                            f"🎯 Take Profit: {take_profit:.8f}\n"
                            f"📊 ATR: {atr_value:.8f}\n\n"
                            f"📈 *Fibonacci TP Levels:*\n"
                            f"• TP1 (23.6%): {fib_tp_levels[0]:.8f}\n"
                            f"• TP2 (61.8%): {fib_tp_levels[1]:.8f}\n"
                            f"• TP3 (78.6%): {fib_tp_levels[2]:.8f}\n\n"
                            f"📊 *Indikator:*\n"
                            f"• RSI: {latest_bar['rsi']:.2f}\n"
                            f"• EMA{EMA_FAST}: {latest_bar['ema_fast']:.8f}\n"
                            f"• EMA{EMA_SLOW}: {latest_bar['ema_slow']:.8f}"
                        )

                        # Send to Telegram
                        send_telegram_message(message)
                        send_telegram_image("Trade Setup Visualization", chart_image)

                        # Add to Google Sheets (unchanged from original)
                        current_live_signals.append({
                            'timestamp': latest_timestamp.strftime('%Y-%m-%d %H:%M:%S'),
                            'symbol': pair,
                            'direction': 'sell',
                            'price': f"{latest_bar['close']:.8f}",
                            'signal_type': 'Short Signal',
                            'RSI': f"{latest_bar['rsi']:.2f}",
                            'EMA_Fast': f"{latest_bar['ema_fast']:.8f}",
                            'EMA_Slow': f"{latest_bar['ema_slow']:.8f}"
                        })
                        sent_signals.add(signal_key)

            if current_live_signals:
                signals_df = pd.DataFrame(current_live_signals)
                append_dataframe_to_sheets(signals_df, GOOGLE_SHEET_NAME, LIVE_SIGNALS_WORKSHEET_NAME)
            else:
                print(f"{datetime.now()} - Tidak ada sinyal baru terdeteksi. Menunggu bar berikutnya...")

            print(f"Tidur selama {sleep_duration.total_seconds() / 3600:.1f} jam ({sleep_seconds} detik) hingga pemeriksaan berikutnya...")
            time.sleep(sleep_seconds)

        except KeyboardInterrupt:
            print("\nMode forward test dihentikan oleh pengguna.")
            break
        except Exception as e:
            print(f"Terjadi error selama loop forward test: {e}. Mencoba lagi dalam 60 detik...")
            time.sleep(60)

    print("\n--- Mode Forward Test Selesai ---")

# --- Main Program ---
if __name__ == "__main__":
    # --- Bagian 1: Backtesting Historis ---
    print("--- Memulai Backtest Historis ---")
    initial_balance = INITIAL_USDT_BALANCE
    risk_per_trade_percent = 0.01

    all_data_frames = []

    for pair in PAIRS:
        try:
            df = fetch_crypto_data_ccxt(pair, TIMEFRAME, limit=1000)
            df = rsi_strategy(df)
            df = atr(df)
            df = calculate_ema(df, EMA_FAST, EMA_SLOW)
            df = candlestick_patterns(df)
            df = generate_signals(df)
            all_data_frames.append(df)
        except Exception as e:
            print(f"Terjadi error saat mengambil data atau menghitung indikator untuk {pair}: {str(e)}")

    if not all_data_frames:
        print("Tidak ada data yang tersedia untuk backtesting. Keluar.")
    else:
        combined_df = pd.concat(all_data_frames).sort_index()

        print("\nMenjalankan backtest global di semua pair...")
        results, final_balance, balance_history_df = backtest(
            combined_df,
            initial_balance=initial_balance,
            risk_per_trade_percent=risk_per_trade_percent,
            atr_multiplier_sl=ATR_MULTIPLIER_SL,
            atr_multiplier_tp=ATR_MULTIPLIER_TP,
            commission_per_trade_percent=COMMISSION_PER_TRADE_PERCENT
        )

        print(f"\n--- Ringkasan Backtest Global ---")
        print(f"Saldo Akhir: ${final_balance:.2f}")
        print(f"Saldo Awal: ${initial_balance:.2f}")
        print(f"Profit Bersih: ${final_balance - initial_balance:.2f}")

        if not results.empty:
            analyzed_results = analyze_results(results, initial_balance)
            plot_balance(balance_history_df, initial_balance)
            print_transactions(results)

            print(f"\nMengekspor transaksi backtest ke Google Sheets...")
            export_cols = ['entry_date', 'exit_date', 'symbol', 'direction', 'entry_price',
                           'exit_price', 'pnl', 'exit_type', 'risk_reward', 'balance', 'commission', 'lot_size_amount']
            export_df = results[export_cols].copy()
            export_df['entry_date'] = export_df['entry_date'].dt.strftime('%Y-%m-%d %H:%M:%S')
            export_df['exit_date'] = export_df['exit_date'].dt.strftime('%Y-%m-%d %H:%M:%S')

            export_dataframe_to_sheets(export_df, GOOGLE_SHEET_NAME, BACKTEST_WORKSHEET_NAME)

        else:
            print("Tidak ada perdagangan yang dieksekusi selama periode backtest.")

    print("\n--- Backtest Historis Selesai ---")

    # --- Bagian 2: Forward Test dengan Telegram ---
    forward_test()

--- Memulai Backtest Historis ---
2025-05-26 01:56:38.663895 - Mengunduh data untuk BTC/USDT dengan ccxt..
2025-05-26 01:56:39.773234 - Mengunduh data untuk ETH/USDT dengan ccxt..
2025-05-26 01:56:39.936621 - Mengunduh data untuk SOL/USDT dengan ccxt..
2025-05-26 01:56:40.088665 - Mengunduh data untuk XRP/USDT dengan ccxt..
2025-05-26 01:56:40.265264 - Mengunduh data untuk SUI/USDT dengan ccxt..
2025-05-26 01:56:40.401043 - Mengunduh data untuk HAEDAL/USDT dengan ccxt..
2025-05-26 01:56:40.564536 - Mengunduh data untuk ADA/USDT dengan ccxt..
2025-05-26 01:56:40.681916 - Mengunduh data untuk DOGE/USDT dengan ccxt..
2025-05-26 01:56:40.816800 - Mengunduh data untuk PEPE/USDT dengan ccxt..
2025-05-26 01:56:40.983824 - Mengunduh data untuk TRUMP/USDT dengan ccxt..
2025-05-26 01:56:41.126913 - Mengunduh data untuk MOODENG/USDT dengan ccxt..
2025-05-26 01:56:41.256385 - Mengunduh data untuk FARTCOIN/USDT dengan ccxt..
2025-05-26 01:56:41.386869 - Mengunduh data untuk HYPE/USDT dengan ccxt..
